In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import plotly.express as px
import matplotlib.colors as mc
import colorsys
import math
import plotly.colors as pc
from shapely.ops import unary_union
from shapely.geometry import shape, mapping
from shapely.ops import nearest_points
from pyproj import Geod
# load excel dataset
raw_data = pd.read_excel(
    'Data/UNHCR_Flow_Data.xlsx',
    sheet_name='DATA'
)

# -------------------------------------------------------------------
#                      changes to the raw data
# -------------------------------------------------------------------

# header to lowercase
raw_data.columns = raw_data.columns.str.lower()
# data to lowercase
for col in raw_data.select_dtypes(include='object'):
    raw_data[col] = raw_data[col].str.lower()

# remove duplicates
raw_data = raw_data.drop_duplicates()

# remove rows with missing values
raw_data = raw_data.dropna()

# ---------------------------------------------------------
#         Dealing with long impractical country names
# ---------------------------------------------------------

# change "originname"
name_edits = {
    "serbia and kosovo: s/res/1244 (1999)": "serbia/kosovo",
    "bolivia (plurinational state of)": "bolivia",
    "iran (islamic rep. of)": "iran",
    "netherlands (kingdom of the)": "nederland",
    "côte d'ivoire": "ivory coast",
    "state of palestine": "palestine",
    "lao people's dem. rep": "laos",
    "russian federation": "russia",
    "syrian arab rep.": "syria",
    "dem. rep. of the congo": "dr congo",
    "congo, republic of": "congo",
    "china, hong kong sar": "hong kong",
    "rep. of korea": "south korea",
    "dem. people's rep. of korea": "north korea",
    "central african rep.": "central african republic",
    "dominican rep.": "dominican republic",
    "united rep. of tanzania": "tanzania",
    "venezuela (bolivarian republic of)": "venezuela",
    "rep. of moldova": "moldova"
}

#            Apply changes
# -----------------------------------
# country name changes to the 'originname' column
raw_data['originname'] = raw_data['originname'].replace(name_edits)
raw_data["asylumname"] = raw_data["asylumname"].replace(name_edits)

# Count rows that have 'count' less than 100
less_than_100 = raw_data[raw_data['count'] < 100].shape[0]
print(f"Number of rows with 'count' less than 100: {less_than_100}")
# filter out rows where 'count' is less than 100
filtered_data = raw_data[raw_data['count'] >= 100].copy()

# export cleaned data to csv
clean_data = raw_data
clean_data.to_csv('Data/cleaned_data.csv', index=False)

# export filtered data to csv
filtered_data.to_csv('Data/filtered_data.csv', index=False)

C:\Users\Loke\AppData\Local\Temp\ipykernel_20852\791625505.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in raw_data.select_dtypes(include='object'):


Number of rows with 'count' less than 100: 76864


In [2]:
# -----------------------------------------------------------------------------------------------------
#                               compute the distances between countries
# -----------------------------------------------------------------------------------------------------

# ----------------------------------
#        Centroid to Centroid
# ----------------------------------

geod = Geod(ellps="WGS84")

# ----------------------------------
#     Centroid to next border
# ----------------------------------

# load country geometries
countries_gdf = gpd.read_file(
    "https://raw.githubusercontent.com/johan/world.geo.json/master/countries.geo.json"
)

# compute distance from origin country centroid to nearest point on border of destination country
# compute distance from origin country centroid
# to nearest point on border of destination country
def distance_to_country_border_km(origin_iso, border_iso, countries_gdf):

    origin_match = countries_gdf.loc[
        countries_gdf["id"].str.upper() == str(origin_iso).upper(),
        "geometry"
    ]

    border_match = countries_gdf.loc[
        countries_gdf["id"].str.upper() == str(border_iso).upper(),
        "geometry"
    ]

    # handle missing ISO codes
    if origin_match.empty or border_match.empty:
        return np.nan

    origin_geom = origin_match.iloc[0]
    border_geom = border_match.iloc[0]

    origin_point = origin_geom.representative_point()

    border_line = border_geom.boundary

    nearest_origin_point, nearest_border_point = nearest_points(
        origin_point,
        border_line
    )

    _, _, distance_m = geod.inv(
        nearest_origin_point.x,
        nearest_origin_point.y,
        nearest_border_point.x,
        nearest_border_point.y,
    )

    return round((distance_m / 1000), 2)

print()
print("centroid to border distances:")
print("distance between central Afghanistan and the border if Iran:", distance_to_country_border_km("afg", "irn", countries_gdf))
print("distance between central Australia and the border of Brazil:", distance_to_country_border_km("aus", "bra", countries_gdf))
print("distance between central France and the border of Germany:", distance_to_country_border_km("fra", "deu", countries_gdf))



centroid to border distances:
distance between central Afghanistan and the border if Iran: 404.83
distance between central Australia and the border of Brazil: 16208.87
distance between central France and the border of Germany: 416.97


In [3]:
# -------------------------------------------------
#        Helper functions for bubble charts
# -------------------------------------------------

def prepare_plot_data(origin_iso, year_start, year_end, min_count=None):
    """Prepare data for refugee distance vs count plots"""
    sub = filtered_data[
        (filtered_data["originiso"] == origin_iso)
        & (filtered_data["year"].between(year_start, year_end))
    ].copy()

    plot_df = (
        sub
        .groupby(["asylumiso", "asylumname"], as_index=False)["count"]
        .sum()
    )

    if min_count:
        plot_df = plot_df[plot_df["count"] >= min_count].copy()

    plot_df["border_distance_km"] = plot_df.apply(
        lambda row: distance_to_country_border_km(
            origin_iso,
            row["asylumiso"],
            countries_gdf
        ),
        axis=1
    )

    return plot_df


def lighten_color_hex(color, amount=0.35):
    """Lightens a hex color using RGB interpolation"""
    r, g, b = pc.hex_to_rgb(color)
    r = int(r + (255 - r) * amount)
    g = int(g + (255 - g) * amount)
    b = int(b + (255 - b) * amount)
    return f"rgb({r},{g},{b})"


def lighten_color_hls(color, amount=0.85):
    """Lightens a color using HLS color space"""
    try:
        c = mc.cnames[color]
    except:
        c = color
    r, g, b = mc.to_rgb(c)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = 1 - amount * (1 - l)
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    return f"rgb({int(r*255)}, {int(g*255)}, {int(b*255)})"


def apply_hover_styling(fig, lighten_func, lighten_amount):
    """Apply hover label styling to all traces"""
    for trace in fig.data:
        original_color = trace.marker.color
        trace.hoverlabel = dict(
            bgcolor=lighten_func(original_color, lighten_amount),
            font_size=13,
            font_color="black",
            bordercolor=original_color,
        )
    return fig


def add_grid_lines(fig, plot_df, x_min, y_min=100):
    """Add dotted grid lines from origin to each point"""
    for _, row in plot_df.iterrows():
        x = row["border_distance_km"]
        y = row["count"]
        color = "#B8B8B8"

        fig.add_shape(
            type="line",
            x0=x_min, x1=x, y0=y, y1=y,
            xref="x", yref="y",
            line=dict(color=color, width=1, dash="dot"),
            layer="above",
        )
        fig.add_shape(
            type="line",
            x0=x, x1=x, y0=y_min, y1=y,
            xref="x", yref="y",
            line=dict(color=color, width=1, dash="dot"),
            layer="above",
        )
    return fig


def ordered_asylumisos_by_label(plot_df):
    """Return asylum ISO codes ordered by the combined ISO/name label used previously."""
    unique_pairs = plot_df[["asylumiso", "asylumname"]].drop_duplicates()
    return [
        iso
        for iso, _ in sorted(
            unique_pairs.itertuples(index=False, name=None),
            key=lambda pair: f"{pair[0].lower()} — {pair[1].lower()}",
        )
    ]


In [4]:
# -------------------------------------------------
#        Number of refugees VS distance
# -------------------------------------------------

# Prep plot data
afg_plot = prepare_plot_data("afg", 1980, 1988)
afg_order = ordered_asylumisos_by_label(afg_plot)

# custom colors
color_discrete_map = {
    "pak": "#28035e",
    "irn": "#9E84FA",
    "deu": "#a855f7",
    "usa": "#c084fc",
    "sau": "#d8b4fe",
    "ind": "#fcabcf",
}

# create scatter plot
fig = px.scatter(
    afg_plot,
    x="border_distance_km",
    y="count",
    size="count",
    color="asylumiso",
    color_discrete_map=color_discrete_map,
    category_orders={"asylumiso": afg_order},
    text="asylumiso",
    log_y=True,
    hover_name="asylumname",
    hover_data={"asylumiso": False},
    labels={
        "border_distance_km": "Distance to border (km) - log scale",
        "count": "Number of Afghan refugees - log scale",
        "asylumiso": "Destination country",
    },
    title="From Afghanistan to imigration country border - by distance and flow size (1980-1988)",
    color_discrete_sequence=px.colors.sequential.Viridis,
)

# put iran trace last so it is drawn on top
fig.data = tuple(
    [trace for trace in fig.data if not trace.name.startswith("irn")]
    + [trace for trace in fig.data if trace.name.startswith("irn")]
)

# style scatter plot
fig.update_traces(
    textposition="top center",
    marker=dict(sizemode="area", sizeref=40, opacity=0.8),
)

apply_hover_styling(fig, lighten_color_hls, 0.35)

# axes
fig.update_yaxes(
    range=[2, 7],
    tickvals=[1e2, 1e3, 1e4, 1e5, 1e6],
    ticktext=["100", "1k", "10k", "100k", "1M"],
)

x_min = -1000
x_max = 5300
fig.update_xaxes(
    range=[x_min, x_max],
    tickvals=[0, 1000, 2000, 3000, 4000, 5000],
    ticktext=["0", "1000", "2000", "3000", "4000", "5000"],
)

fig.update_layout(
    title_x=0.5,
    width=1200,
    height=900,
    plot_bgcolor="#eeeeee",
    paper_bgcolor="white",
    showlegend=False,
    font=dict(family="Helvetica", size=13),
    margin=dict(l=120, r=80, t=100, b=100),
)

add_grid_lines(fig, afg_plot, x_min)

# save plot
fig.write_html("../visualizations/afghanistan_distance_vs_count.html")

fig.show()


In [5]:
# -------------------------------------------------
#     Number of refugees VS distance
#                1994-1996
# -------------------------------------------------

# Prep plot data
rwanda_plot = prepare_plot_data("rwa", 1994, 1996)
rwanda_order = ordered_asylumisos_by_label(rwanda_plot)

# create scatter plot
fig = px.scatter(
    rwanda_plot,
    x="border_distance_km",
    y="count",
    size="count",
    color="asylumiso",
    category_orders={"asylumiso": rwanda_order},
    hover_data={"asylumiso": False},
    text="asylumiso",
    log_y=True,
    hover_name="asylumname",
    labels={
        "border_distance_km": "Distance to border (km) - log scale",
        "count": "Number of Rwanda refugees - log scale",
        "asylumiso": "Destination country",
    },
    title="From Rwanda to imigration country border - by distance and flow size (1994-1996)",
    color_discrete_sequence=px.colors.sequential.Viridis,
)

# put Burundi and Tanzania on top
fig.data = tuple(
    [trace for trace in fig.data
     if not (trace.name.startswith("bdi") or trace.name.startswith("tza"))]
    + [trace for trace in fig.data if trace.name.startswith("tza")]
    + [trace for trace in fig.data if trace.name.startswith("bdi")]
)

# style scatter plot
fig.update_traces(
    textposition="top center",
    marker=dict(sizemode="area", sizeref=15, opacity=0.6),
)

apply_hover_styling(fig, lighten_color_hls, 0.35)

# axes
fig.update_yaxes(
    range=[2, 7],
    tickvals=[1e2, 1e3, 1e4, 1e5, 1e6],
    ticktext=["100", "1k", "10k", "100k", "1M"],
)

x_min = -1000
x_max = 6500
fig.update_xaxes(
    range=[x_min, x_max],
    tickvals=[0, 1000, 2000, 3000, 4000, 5000, 6000],
    ticktext=["0", "1000", "2000", "3000", "4000", "5000", "6000"],
)

fig.update_layout(
    title_x=0.5,
    width=1200,
    height=900,
    plot_bgcolor="#f0f0f0",
    paper_bgcolor="white",
    showlegend=False,
    font=dict(family="Helvetica", size=13),
    margin=dict(l=120, r=80, t=100, b=100),
)

add_grid_lines(fig, rwanda_plot, x_min)

# save and show
fig.write_html("../visualizations/rwanda_distance_vs_count.html")

fig.show()


In [6]:
# -------------------------------------------------
#        Ukraine war: refugees VS distance
#        2022-2024
# -------------------------------------------------

# Prep plot data
ukraine_plot = prepare_plot_data("ukr", 2022, 2025, min_count=1000)
ukraine_order = ordered_asylumisos_by_label(ukraine_plot)

# create scatter plot
fig = px.scatter(
    ukraine_plot,
    x="border_distance_km",
    y="count",
    size="count",
    log_x=True,
    log_y=True,
    color="asylumiso",
    hover_data={"asylumiso": False},
    category_orders={"asylumiso": ukraine_order},
    text="asylumiso",
    hover_name="asylumname",
    labels={
        "border_distance_km": "Distance to destination border (km) - log scale",
        "count": "Number of Ukrainian refugees - log scale",
        "asylumiso": "Destination country",
    },
    title="From Ukraine to imigration country border - by distance and flow size (2022-2025)",
    color_discrete_sequence=px.colors.sequential.Viridis,
)

# put major nearby destinations on top
fig.data = tuple(
    [trace for trace in fig.data
     if not (trace.name.startswith("pol") or trace.name.startswith("deu") or trace.name.startswith("cze"))]
    + [trace for trace in fig.data if trace.name.startswith("cze")]
    + [trace for trace in fig.data if trace.name.startswith("deu")]
    + [trace for trace in fig.data if trace.name.startswith("pol")]
)

# style scatter plot
fig.update_traces(
    textposition="top center",
    marker=dict(sizemode="area", sizeref=120, opacity=0.7),
)

apply_hover_styling(fig, lighten_color_hex, 0.65)

# axes
fig.update_yaxes(
    tickvals=[5e3, 1e4, 5e4, 1e5, 3e5, 5e5, 7e5, 1e6, 1.5e6, 2.e6],
    ticktext=["5K", "10K", "50K", "100K", "300K", "500K", "700K", "1M", "1.5M", "2M"],
)

fig.update_xaxes(
    range=[math.log10(250), 4],
    tickvals=[250, 500, 1000, 2000, 5000],
    ticktext=["250", "500", "1000", "2000", "5000"],
)

fig.update_layout(
    title_x=0.5,
    width=1200,
    height=900,
    plot_bgcolor="#f0f0f0",
    paper_bgcolor="white",
    showlegend=False,
    font=dict(family="Helvetica", size=13),
    margin=dict(l=120, r=80, t=100, b=100),
)

# add reference lines
reference_lines = [
    (5_000, "5K"),
    (10_000, "10K"),
    (50_000, "50K"),
    (100_000, "100K"),
    (300_000, "300K"),
    (500_000, "500K"),
    (700_000, "700K"),
    (1_000_000, "1M"),
    (1_500_000, "1.5M"),
    (2_000_000, "2M"),
]

for y_value, label in reference_lines:
    fig.add_hline(
        y=y_value,
        line_width=1,
        line_dash="dot",
        line_color="#BDBDBD",
        annotation_text=label,
        annotation_position="left",
        annotation_font_size=11,
        annotation_font_color="#555555",
    )

# add extra scatter for readability
for trace in fig.data:
    try:
        y = trace.y[0]
    except:
        continue

    if y > 50_000:
        fig.add_scatter(
            x=trace.x,
            y=trace.y,
            mode="markers",
            marker=dict(size=6, color=trace.marker.color),
            showlegend=False,
            hoverinfo="skip",
        )

# save and show
fig.write_html("../visualizations/ukraine_distance_vs_count.html")

fig.show()

